# Lekcija 18: Zavarovanje AI agentov s kriptografskimi prejemki

## Interaktivni zvezek

Ta zvezek vas vodi skozi štiri vaje:

1. **Podpišite svoj prvi prejem** za klic orodja agenta in ga preverite.
2. **Spremeni prejem** in opazuj, kako preverjanje ne uspe.
3. **Zgradi verigo iz treh prejemkov** in potrdi celovitost verige.
4. **Ovij klic orodja Microsoft Agent Framework**, tako da vsak ukrep ustvari prejemek.

Vse kriptografske primitivke so uvožene iz dobro vzdrževanih knjižnic (`pynacl` za Ed25519, `jcs` za RFC 8785 kanonični JSON, `hashlib` iz standardne Python knjižnice za SHA-256). Logika prejemka sama je preprost Python, ki ga lahko prebereš in spremeniš.

Zaženi celice v vrstnem redu. Vsak odsek je kratek in samostojen.


## Namestitev

Namestite obe odvisnosti. Obe imata permisivne licence (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Pomočna orodja

Ta dva pomočnika obdelujeta kodiranje base64url (brez dopolnitve) in hasiranje SHA-256 poljubnih objektov. Omogočata, da je preostanek zvezka osredotočen na samo logiko računa.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Oddelek 1: Podpišite svoj prvi račun

Predstavljajte si, da je naš agent za **Contoso Travel** pravkar poiskal lete iz Sydneyja v Los Angeles za stranko. Želimo zabeležiti izjavo prehoda o tem klicu orodja, da jo lahko prihodnji revizor preveri pod ločeno pripetim ključem izdajatelja.

### Korak 1.1: Ustvarite podpisni ključ

V produkciji bi agentov podpisni ključ živel v strojno varnem modulu (HSM), Azure Key Vault ali podobnem zaščitenem skladišču. Za to lekcijo ustvarimo svež ključ v pomnilniku.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Korak 1.2: Ustvarite vsebino potrdila

Vsebina vsebuje vse, čemu želimo, da potrdilo priča: kdo je deloval, katero orodje, s katerimi argumenti, kaj je bilo vrnjeno, pod katero politiko in kdaj. Argumente in rezultat zgoščujemo, namesto da bi jih vstavili neposredno, da potrdilo ne razkrije občutljive vsebine.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Korak 1.3: Podpišite in sestavite potrdilo

Tri koraki:

1. Kanonizirajte nosilec podatkov z uporabo JCS, tako da dve implementaciji, ki proizvedeta isti logični račun, proizvedeta bajtno identične bajte.
2. Podpišite kanonične bajte neposredno z zasebnim ključem Ed25519. PureEdDSA interno zgošča sporočilo, zato bi dodaten predhodni zgoščevanje spremenilo protokol.

Podpis je nato priložen originalnemu nosilcu podatkov, da se ustvari končno potrdilo.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Korak 1.4: Preveri prejem

Preverjanje obrne postopek. Odstranimo podpis, ponovno izračunamo kanonične bajte in preverimo podpis glede na javni ključ izdajatelja.

Preverjanje poteka brez povezave: ni treba klicati nobene storitve in ni omrežnega dostopa. Potrebuje pa eno stvar poleg prejema: javni ključ izdajatelja, pridobljen preko kanala, ki mu zaupa revizor, na primer niz JWK, ki ga izdajatelj objavi, ali vrednost, vnaprej določena. Polje `public_key` znotraj prejema ne more igrati te vloge, ker tisti, ki napiše prejem, napiše tudi to polje. Poglavje 2 predstavi ponarejanje, ki mu to prepreči.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Morali bi videti `Receipt is valid: True`, `Pre-hashed receipt valid: False` in `Malformed receipts rejected: True`. Pozitivni primer dokazuje, da pot neposrednega podpisovanja JCS deluje; negativni nadzor omogoča izvajanje pravila obsega podpisa namesto, da ostane kot proza; zadnja vrstica prikazuje nepravilno oblikovan vhod, ki vrača `False` namesto, da bi sprožil izjemo.


## Oddelek 2: Spreminjanje računa

Celoten namen računov je, da je vidno, če so bili spremenjeni. Dokažimo to.

Spremenili bomo natanko en znak na računu in opazovali, kako preverjanje ne uspe.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Kaj se je pravkar zgodilo?

Ko smo spremenili `policy_id`, so se spremenili kanonični bajti. Podpis (ki je bil nad izvirnimi kanoničnimi bajti) se ne ujema več. Preverjanje pravilno vrne `False`.

Ni načina, da bi spremenili podpisano polje, torej karkoli izven objekta `signature`, in še vedno imeli potrdilo, ki se preveri proti javnemu ključu izdajatelja, razen če napadalec nima zasebnega ključa izdajatelja. Sam objekt `signature` ni podpisan: `sig` je podpis, in `verify_receipt` zanemarja `public_key`, tako da popravek samo tega polja ne spremeni ničesar. Dokler je zasebni ključ v ključni shrambi in preverjevalci uporabljajo neodvisno objavljen ključ izdajatelja, je sprememba podpisanega polja nemogoče skriti. Ta pogoj je pomemben: naslednja celica pokaže spremenjeno potrdilo, ki se popolnoma preveri proti drugemu ključu.

Poskusi sam: spremeni `tool_name` ali `agent_id` ali `timestamp` v zgornji celici in znova zaženi. Vsaka sprememba podpisanega polja ustvari neveljavno potrdilo.


### Neavtorizirana sprememba, ki je podpis ne zazna

Zgornji napadalec ni ponovno podpisal. Bolj pazljiv bi: spremenil prejemek,
ga podpisal z lastnim ključem, ki ga je sam ustvaril, in ta ključ
vnesel v polje `public_key`. Rezultat je prejemek, katerega podpis je popolnoma verodostojen
glede na njegovo vsebino.

Zaženi to in poglej, katera od dveh preverjanj to odkrije.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Moral bi videti `True` nato `False`. Tri vrstice `assert` naredijo to za
preverjanje: celica sproži napako, če ponaredek kdaj preveri proti
izdajateljevemu ključu.

Ponaredek je interno skladen. Njegov podpis res ustreza njegovi
vsebini in ključu, ki ga nosi, zato preverjalnik, ki prebere ključ iz
potrdila, poroča, da je veljavno. Ničesar na potrdilu se ne zdi narobe, ker
ni ničesar nepravilnega.

Edino, kar ga ujame, je preverjanje proti ključu, ki ga napadalec ni mogel
vplivati. Zato `verify_receipt` sprejme ključ kot argument: funkcije s podpisom
`verify_receipt(receipt)` ni mogoče varno napisati,
ker je edini ključ, ki ga lahko doseže, tisti, ki ga je podal napadalec.

Enako potrdilo se shrani kot `sample_receipts/04_resigned_receipt.json`, če ga želiš
pregledati zunaj zvezka.


## Poglavje 3: Povežite potrdila skupaj

Ena potrdilo ščiti eno dejanje. Večina agentov izvede veliko dejanj. Da naredimo celotno zaporedje vidno za posege, povežemo vsako potrdilo s prejšnjim tako, da vključimo zgoščeno vrednost prejšnjega potrdila v vsebino novega potrdila.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Če kdo odstrani ali preuredi potrdilo znotraj prikazanega zaporedja, naslednja povezava ne bo več veljala. Za zaznavanje okrajšave ali prepisovanja z nadzorom podpisnika je potrebna tudi zavarovana trenutna glava, zunanji kontrolni točka ali overjena zaveza populacije.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Zdaj pretrgajte verigo s poseganjem v srednji prejemek in ponovno preverite. Spremenjeni prejemek ne uspe pri preverjanju podpisa, IN naslednji prejemek ne uspe pri preverjanju verižne povezave (ker njegov `previous_receipt_hash` ne ustreza več spremenjenemu zgoščevanju srednjega prejemka).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Prejemek 0 se še vedno preverja (ni bil spremenjen in nima predhodnika, od katerega bi bil odvisen). Prejemek 1 ne uspe preverjanja podpisa, ker smo spremenili `tool_args_hash`. Prejemek 2 ne uspe preverjanja verige povezav, ker je bil njegov `previous_receipt_hash` izračunan glede na izvirni (sedaj spremenjeni) prejemek 1.

Tudi če napadalec ponovno podpiše spremenjeni prejemek 1, novi podpis ne uspe glede na ključ izdajatelja, razen če ima napadalec zasebni ključ izdajatelja (ponovni podpis s svojim ključem ne uspe, kot je pokazal 2. razdelek), in neusklajenost verige povezav v prejemku 2 bi še vedno razkrila spremembo. Da bi skrili spremembo, bi moral napadalec ponovno podpisati vsak prejemek od točke spremembe naprej, kar zahteva posedovanje zasebnega ključa izdajatelja.


## Poglavje 4: Ovijte klic orodja agenta s podpisom potrdila

V dejanski uvedbi ne želite, da si vsak avtor agenta zapomni klicati `make_receipt`. Želite, da je podpis potrdila samodejen za vsak klic orodja.

Tukaj je najpreprostejši vzorec: ovojnica razreda, ki sprejme katerokoli klicno funkcijo orodja in vrne različico, ki oddaja potrdila. Ta se prilagaja kateremukoli agentnemu ogrodju, vključno z Microsoft Agent Framework (`agent_framework.foundry`).

Če nimate nastavljenega Microsoft Foundry projekta, lokalni ponaredek spodaj še vedno prikazuje ta vzorec.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integracija z Microsoft Agent Frameworkom

Zgornji zavijalnik `ReceiptedTool` je neodvisen od ogrodja. Če ga želite uporabiti znotraj agenta, zgrajenega z Microsoft Agent Frameworkom, registrirajte zavito funkcijo kot orodje. Osnutek (mock bi nadomestili z dejansko registracijo orodja Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agentno ogrodje ne potrebuje vedeti ničesar o potrdilih. Podpisovanje potrdil je zavito okoli orodja, ne vgrajeno v ogrodje. Tako dodate izvor obstoječi kodi agenta brez ponovnega pisanja agenta.


## Povzetek in izziv za razširitev

Imate:

- Ustvarjen par ključev Ed25519.
- Sestavljen in podpisan prejemek za klic orodja agenta.
- Preverili prejemek brez povezave z uporabo samo javnega ključa izdajatelja, pridobljenega neodvisno od prejemka.
- Sprememba prejemka in opazovanje neuspeha pri preverjanju.
- Sestavljena veriga treh prejemkov z uporabo verižnega zgoščevanja.
- Sprememba sredine verige in opazovanje neuspeha tako podpisa kot povezave verige.
- Ovijanje funkcije orodja z avtomatskim podpisom prejemkov.

**Izziv za razširitev.** Razširite shemo prejemka z `request_id` poljem (UUID za porazdeljeno sledenje). Posodobite `make_receipt`, da ga vključuje, in potrdite, da se prejemki še vedno potrjujejo od začetka do konca. Nato spremenite polje po podpisu in potrdite, da preverjanje ne uspe. To vas prisili, da razumete, kako vsak bajt kanoničnega kodiranja prispeva k podpisu.

**Pomembna meja.** Pod pripetim ključem in kontekstom verige prejemki dokazujejo pripis podpisa, integriteto vsebine in povezavo s predhodnikom, ki je bil predstavljen. Ne dokazujejo izvajanja, resnice vira, zaupanja vrednega časa, popolnosti, ocenjevanja pravilnikov ali pravilnosti. Prejemki so temelj. Upravljanje je sistem, ki ga zgradite na tem temelju.

Ponovno preberite README lekcije s to mejo v mislih. Najpogostejša napaka ekip pri prejemkih je domneva, da "imamo prejemke" pomeni "smo upravljani." Ni tako. Prejemki omogočajo revizijo vedenja agenta. Ne zagotavljajo njegove pravilnosti.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
